<a href="https://colab.research.google.com/github/maick-code/AIMS-Capstone/blob/arena%2F01a03c0c-aims-capstone/VaxiMere_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VaxiMère-QA-CG — Pipeline complet (Google Colab)

Construction du dataset d'intentions multilingue (FR / lingala / kituba) pour la
vaccination pédiatrique au Congo-Brazzaville.

**Étapes** : 1) install 2) vérif GPU 3) récupération du code (clone auto)
4) test rapide (`dryrun`, sans modèle) 5) exécution complète 6) inspection 7) téléchargement.

> Exécutez les cellules **dans l'ordre**. La cellule 3 clone automatiquement le dépôt
> depuis GitHub ; aucune action manuelle n'est nécessaire.

In [37]:
# 1) Installation des dépendances (Colab fournit déjà torch/transformers)
!pip install -q datasets transformers pandas accelerate sentencepiece huggingface_hub

In [38]:
# 2) Vérification du GPU (un T4 suffit largement)
!nvidia-smi -L
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU 0: Tesla T4 (UUID: GPU-642a449a-b750-ad91-095d-79a09ac45c3d)
CUDA disponible : True
Device : Tesla T4


In [39]:
# 3) Récupération du code — clone AUTOMATIQUE (avec fallback)
#    Si le dépôt est déjà présent (ou les fichiers uploadés à la racine), on le réutilise.
import os, subprocess
from pathlib import Path
from IPython import get_ipython

REPO_URL = "https://github.com/maick-code/AIMS-Capstone.git"
BRANCH   = "arena/01a03c0c-aims-capstone"   # branche qui contient le pipeline
REPO_DIR = Path("/content/AIMS-Capstone")

def find_repo():
    for c in (REPO_DIR, Path("/content"), Path.cwd()):
        if (c / "run_pipeline.py").exists() and (c / "vaximere").exists():
            return c
    return None

repo = find_repo()
if repo is None:
    # essai 1 : branche du pipeline ; essai 2 : branche par défaut
    for args in (["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
                 ["git", "clone", REPO_URL, str(REPO_DIR)]):
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode == 0:
            break
    repo = find_repo()

if repo is None:
    print("⚠️ Échec du clonage automatique.")
    print("👉 Uploadez manuellement dans Colab : run_pipeline.py, vaximere/,")
    print("   selftest.py et DATA_CARD.md, puis relancez cette cellule.")
    repo = Path.cwd()
else:
    get_ipython().run_line_magic("cd", str(repo))
    print("✅ Dépôt prêt :", repo)

print("Répertoire de travail :", os.getcwd())
print("Fichiers :", sorted(f for f in os.listdir(repo) if not f.startswith('.')))

/content/AIMS-Capstone
✅ Dépôt prêt : /content/AIMS-Capstone
Répertoire de travail : /content/AIMS-Capstone
Fichiers : ['DATA_CARD.md', 'LICENSE', 'README.md', 'VaxiMere_Colab.ipynb', 'data', 'requirements.txt', 'run_pipeline.py', 'selftest.py', 'vaximere']


In [40]:
# 4) Test rapide SANS modèle ni réseau (valide tout le câblage du pipeline)
#    Sorties : data/dryrun/ (aucun téléchargement de modèle)
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python selftest.py
!python run_pipeline.py --mode dryrun

[OK ] 8 intentions définies
[OK ] Mapping FAQ couvre les 8 intentions
[OK ] 3 langues définies (fra/lin/mkw)
[OK ] query_id format
[OK ] query_id suffixe lingala
[OK ] query_id suffixe kituba
[OK ] mots-clés de domaine non vides
[OK ] domaine match : quand vacciner mon bébé contre la rougeo...
[OK ] domaine match : le carnet vaccinal de mon enfant est per...
[OK ] domaine match : mon bébé a de la fièvre après le vaccin ...
[OK ] domaine match : le BCG protège contre la tuberculose...
[OK ] hors domaine ne matche pas les mots-clés vaccinaux
[OK ] 8 hypothèses zero-shot
[OK ] candidate_labels alignés
[OK ] mapping label->intention bijectif
[OK ] banque seed : 272 questions
[OK ] aucun doublon exact
[OK ] seed UTILITE_VACCIN : >= 30 questions
[OK ] seed SECURITE_VACCIN : >= 30 questions
[OK ] seed CALENDRIER_RDV : >= 30 questions
[OK ] seed RETARD_RATTRAPAGE : >= 30 questions
[OK ] seed EFFET_SECONDAIRE : >= 30 questions
[OK ] seed RUMEUR_CROYANCE : >= 30 questions
[OK ] seed LOCALISATION

In [ ]:
# 5) Exécution COMPLÈTE (télécharge mDeBERTa-v3 + NLLB-600M, exécute le pipeline)
#    Durée estimée sur T4 : ~10-20 min selon la connexion.
#    Sorties : data/final/
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python run_pipeline.py --mode full

04:23:39 | INFO    | === Pipeline VaxiMère-QA-CG (mode=full) ===
04:23:39 | INFO    | >>> Étape 0 — Banque seed ...
04:23:39 | INFO    | Distribution seed (avant filtrage) :
intention
CALENDRIER_RDV           34
EFFET_SECONDAIRE         34
HORS_DOMAINE_CLINIQUE    34
LOCALISATION_ACCES       34
RETARD_RATTRAPAGE        34
RUMEUR_CROYANCE          34
SECURITE_VACCIN          34
UTILITE_VACCIN           34
04:23:39 | INFO    | <<< Étape 0 — Banque seed terminé en 0.0s
04:23:39 | INFO    | >>> Étape 1 — Extraction Hugging Face ...
04:23:39 | INFO    | >>> Extraction FrenchMedMCQA ...
04:23:40 | WARNING | Échec du chargement de qanastek/frenchmedmcqa : Dataset scripts are no longer supported, but found frenchmedmcqa.py
04:23:40 | WARNING | Échec du chargement de qanastek/frenchmedmcqa : Dataset scripts are no longer supported, but found frenchmedmcqa.py
04:23:40 | WARNING | Échec du chargement de qanastek/frenchmedmcqa : Dataset scripts are no longer supported, but found frenchmedmcqa.py
0

In [ ]:
# 6) Inspection des sorties (data/final/ après `--mode full`, data/dryrun/ après le test)
import json
from pathlib import Path

for base in ("data/final", "data/dryrun"):
    d = Path(base)
    if not d.exists():
        continue
    print(f"== {base} ==")
    for f in sorted(d.glob("*.jsonl")):
        n = sum(1 for _ in f.open(encoding="utf-8"))
        first = next(f.open(encoding="utf-8"), "").strip()
        print(f"  {f.name}: {n} lignes")
        print(f"    aperçu -> {first[:180]}")
    stats = d / "stats_report.json"
    if stats.exists():
        s = json.load(stats.open(encoding="utf-8"))
        print("  stats_report.json :")
        print(json.dumps(s, ensure_ascii=False, indent=2)[:1500])

In [ ]:
# 7) Téléchargement des livrables (DATA_CARD.md est à la racine, pas dans data/)
from google.colab import files
from pathlib import Path

targets = [
    Path("data/final/vaximere_qa_cg_train.jsonl"),
    Path("data/final/faq_validee.json"),
    Path("data/final/stats_report.json"),
    Path("data/final/vaximere_qa_cg_train.csv"),
    Path("DATA_CARD.md"),
]
found = False
for p in targets:
    if p.exists():
        files.download(str(p))
        found = True
    else:
        print("(absent)", p)
if not found:
    print("Aucun livrable trouvé : lancez d'abord `--mode full` (cellule 5).")

In [ ]:
# 8) Pousser le dataset vers le Hugging Face Hub (après la cellule 5 : --mode full)
#    1. Créez un token "write" ici : https://huggingface.co/settings/tokens
#    2. Collez-le quand demandé : il ne sera PAS affiché.
#    3. Le script génère le dataset (parquet) + faq_validee.json + stats + dataset card.
import os, pathlib
from getpass import getpass

os.chdir("/content/AIMS-Capstone")
assert pathlib.Path("data/vaximere_qa_cg_train.jsonl").exists(), \
    "Lancez d'abord la cellule 5 (--mode full) pour générer le dataset."

REPO_ID = "Semence/vaximere-qa-cg"   # <-- adaptez : votre-org/nom-du-dataset

# Token "write" (saisi en mode masqué, jamais enregistré dans le notebook)
os.environ["HF_TOKEN"] = getpass("Collez votre token Hugging Face (write) : ")

!python push_to_hub.py --repo-id {REPO_ID}
print("\n✅ Dataset publié : https://huggingface.co/datasets/" + REPO_ID)